# affect_aif Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/har5h1l/affect_aif/blob/master/notebooks/demo.ipynb)

Run small, Colab-compatible analogues of the numbered paper experiments and inspect the generated outputs. The demo configs mirror `configs/paper/01_*` through `05c_*`, but use fewer seeds, shorter episodes, and reduced profile grids.

Default behavior writes scratch outputs under `outputs/notebook_demo/`, not canonical `results/`, so your clean result packet remains shareable.


## What This Notebook Proves

This is a run-first notebook, not a precomputed-results viewer. Each section runs a demo TOML config through `scripts/experiment/run.py`, analyzes fresh `results.csv` files, and plots a compact payoff/entropy readout.

The default path runs the main paper-spine demos. Profile-factorial and forgiveness demos are marked optional because they expand to more runs even at demo scale. Use `notebooks/reproduce.ipynb` when you want the full paper configs and canonical `results/paper/` materialization.


## 1. Bootstrap Runtime

In Colab, clone the selected branch and move into the repo. Locally, use whatever working directory you started the notebook from; the run cells use the same relative paths as the CLI.


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except Exception:
    IN_COLAB = Path("/content").exists()

def sanitize_repo_url(value):
    value = str(value).strip()
    if value.startswith("[") and "](" in value and value.endswith(")"):
        value = value.split("](", 1)[1][:-1]
    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1]
    return value.strip()


REPO_URL = sanitize_repo_url(os.environ.get("AFFECT_AIF_REPO_URL", "https://github.com/har5h1l/affect_aif.git"))
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "master")

if IN_COLAB:
    repo_dir = Path("/content/affect_aif")
    if repo_dir.exists() and not (repo_dir / ".git").exists():
        shutil.rmtree(repo_dir)
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

print("Working directory:", Path.cwd())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 2. Install Dependencies

Colab runtimes are fresh, so install the repo there by default. Local users can set `INSTALL_DEPS = True` if their kernel does not already have the package installed.


In [ ]:
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install. Set INSTALL_DEPS = True if imports fail.")


## 3. Check Runtime Device

The trust-task demos do not require a GPU, but this cell reports the best available accelerator. JAX will use visible GPU devices automatically when the runtime provides them.


In [ ]:
def detect_accelerator():
    try:
        import jax
        devices = [str(device) for device in jax.devices()]
    except Exception as exc:
        return "gpu" if shutil.which("nvidia-smi") else "cpu", [f"JAX unavailable: {exc}"]

    gpu_devices = [device for device in devices if "gpu" in device.lower() or "cuda" in device.lower()]
    if gpu_devices:
        return "gpu", devices
    return "cpu", devices

DEVICE, JAX_DEVICES = detect_accelerator()
print("Selected device:", DEVICE)
print("JAX devices:", JAX_DEVICES)

if DEVICE == "gpu" and shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
elif DEVICE == "cpu":
    print("No GPU detected; the demo will run on CPU.")


## 4. Demo Controls And Helpers

`run_demo(name)` is the main helper. By default, top-to-bottom execution runs only `predictability_value` so the notebook proves the install-run-analyze path quickly. Add names to `SELECTED_DEMOS`, or set it to `"all"`, when you want more sections to execute.


In [ ]:
RUN_DEMO = True
RUN_ANALYSIS = True
RUN_OPTIONAL_DEMOS = False
RESET_OUTPUTS = False
OUTPUT_ROOT = Path("outputs")
DEMO_BATCH_PREFIX = "notebook_demo"
SELECTED_DEMOS = ["predictability_value"]

DEMO_EXPERIMENTS = {
    "predictability_value": {
        "config": "configs/demo/01_predictability_value.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_01_predictability_value",
        "paper_config": "configs/paper/01_predictability_value.toml",
        "label": "1. Predictability over value",
        "optional": False,
    },
    "deployment_ablation": {
        "config": "configs/demo/02_deployment_ablation.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_02_deployment_ablation",
        "paper_config": "configs/paper/02_deployment_ablation.toml",
        "label": "2. Deployment ablation",
        "optional": False,
    },
    "partner_selection": {
        "config": "configs/demo/03_partner_selection.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_03_partner_selection",
        "paper_config": "configs/paper/03_partner_selection.toml",
        "label": "3. Partner selection",
        "optional": False,
    },
    "betrayal_adaptation": {
        "config": "configs/demo/04_betrayal_adaptation.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_04_betrayal_adaptation",
        "paper_config": "configs/paper/04_betrayal_adaptation.toml",
        "label": "4. Betrayal adaptation",
        "optional": False,
    },
    "alpha_sweep": {
        "config": "configs/demo/05a_alpha_sweep.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_05a_alpha_sweep",
        "paper_config": "configs/paper/05a_alpha_sweep.toml",
        "label": "5a. Alpha sweep",
        "optional": False,
    },
    "prior_factorial": {
        "config": "configs/demo/05b_prior_factorial.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_05b_prior_factorial",
        "paper_config": "configs/paper/05b_prior_factorial.toml",
        "label": "5b. Prior x gain factorial",
        "optional": True,
    },
    "forgiveness": {
        "config": "configs/demo/05c_forgiveness.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_05c_forgiveness",
        "paper_config": "configs/paper/05c_forgiveness.toml",
        "label": "5c. Forgiveness",
        "optional": True,
    },
}

for spec in DEMO_EXPERIMENTS.values():
    if not Path(spec["config"]).exists():
        raise FileNotFoundError(spec["config"])
    if not Path(spec["paper_config"]).exists():
        raise FileNotFoundError(spec["paper_config"])

if RESET_OUTPUTS:
    for spec in DEMO_EXPERIMENTS.values():
        batch_dir = OUTPUT_ROOT / spec["batch"]
        if batch_dir.exists():
            shutil.rmtree(batch_dir)

import pandas as pd
import matplotlib.pyplot as plt


def selected_demo_names():
    if SELECTED_DEMOS == "all":
        return list(DEMO_EXPERIMENTS)
    return list(SELECTED_DEMOS)


def run_required(cmd, required_path=None):
    if required_path is not None and not Path(required_path).exists():
        raise FileNotFoundError(required_path)
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)


def result_paths_for_batch(batch_name):
    batch_dir = OUTPUT_ROOT / batch_name
    return sorted(batch_dir.glob("**/results.csv"))


def run_demo_experiment(name):
    spec = DEMO_EXPERIMENTS[name]
    if name not in selected_demo_names():
        print(f"Skipping {spec['label']}. Add {name!r} to SELECTED_DEMOS to run it.")
        return []
    if spec.get("optional") and not RUN_OPTIONAL_DEMOS:
        print(f"Skipping optional demo: {spec['label']}. Set RUN_OPTIONAL_DEMOS = True to run it.")
        return []

    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        spec["config"],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        spec["batch"],
        "--workers",
        "1",
    ]
    if RUN_DEMO:
        run_required(cmd)
    else:
        print("Experiment run skipped. Set RUN_DEMO = True to execute", name)

    paths = result_paths_for_batch(spec["batch"])
    if not paths:
        raise RuntimeError(f"No results.csv files found for {name} under {OUTPUT_ROOT / spec['batch']}.")
    if RUN_ANALYSIS:
        for result_path in paths:
            run_required(
                [
                    sys.executable,
                    "scripts/analysis/analyze.py",
                    "--results",
                    result_path,
                    "--output-dir",
                    result_path.parent / "notebook_analysis",
                ],
                required_path=result_path,
            )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)
    return paths


def load_results(paths):
    frames = []
    for path in paths:
        frame = pd.read_csv(path, low_memory=False)
        frame["source_file"] = str(path)
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def summarize_by_variant(frame):
    if frame.empty:
        return frame
    metrics = {"payoff": ["mean", "sum"]}
    if "q_pi_entropy" in frame.columns:
        metrics["q_pi_entropy"] = ["mean"]
    return frame.groupby(["experiment_id", "variant_id"], as_index=False).agg(metrics)


def plot_demo_summary(frame, title):
    if frame.empty:
        print(f"No rows available for {title}.")
        return
    summary = frame.groupby("variant_id", as_index=False).agg(
        mean_payoff=("payoff", "mean"),
        total_payoff=("payoff", "sum"),
        mean_entropy=("q_pi_entropy", "mean") if "q_pi_entropy" in frame.columns else ("payoff", "mean"),
    )
    display(summary)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].bar(summary["variant_id"], summary["mean_payoff"])
    axes[0].set(title=f"{title}: mean payoff", xlabel="Variant", ylabel="Payoff")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(summary["variant_id"], summary["mean_entropy"])
    axes[1].set(title=f"{title}: mean entropy", xlabel="Variant", ylabel="Entropy")
    axes[1].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    if "agg" not in plt.get_backend().lower():
        plt.show(block=False)
    plt.close(fig)


def run_demo(name):
    spec = DEMO_EXPERIMENTS[name]
    print(f"Demo config: {spec['config']}")
    print(f"Paper analogue: {spec['paper_config']}")
    paths = run_demo_experiment(name)
    frame = load_results(paths)
    if frame.empty:
        return frame
    print("Rows:", len(frame))
    display(summarize_by_variant(frame))
    plot_demo_summary(frame, spec["label"])
    return frame


## 5. Predictability-Value Demo: Run And Analyze

Short analogue of Section 3.1. It uses the same graded agent-choice setup and affect/global/no-affect contrast as the paper, but with 2 seeds and 40 rounds.


In [ ]:
predictability_value_results = run_demo("predictability_value")


## 6. Deployment-Ablation Demo: Run And Analyze

Short analogue of Section 3.2. It keeps the tracked-only lesion and no-epistemic control but drops to 1 seed and 40 rounds.


In [ ]:
deployment_ablation_results = run_demo("deployment_ablation")


## 7. Partner-Selection Demo: Run And Analyze

Short analogue of Section 3.3. It checks whether the agent-choice pathway runs and exposes partner-selection summaries at demo scale.


In [ ]:
partner_selection_results = run_demo("partner_selection")


## 8. Betrayal-Adaptation Demo: Run And Analyze

Short analogue of Section 3.4. The stance switch happens earlier so the same adaptation path can be inspected quickly.


In [ ]:
betrayal_adaptation_results = run_demo("betrayal_adaptation")


## 9. Alpha-Sweep Demo: Run And Analyze

Short analogue of the Exp A profile sweep. It keeps open-graded and betrayal sub-experiments but uses three alpha values, 1 seed, and 60 rounds.


In [ ]:
alpha_sweep_results = run_demo("alpha_sweep")


## 10. Optional Prior-Factorial Demo: Run And Analyze

Optional analogue of Exp B. It keeps the open, betrayal, and partner-choice sub-experiments, but uses a reduced profile grid. Set `RUN_OPTIONAL_DEMOS = True` to run it.


In [ ]:
prior_factorial_results = run_demo("prior_factorial")


## 11. Optional Forgiveness Demo: Run And Analyze

Optional analogue of Exp C. It shortens the betrayal/repair schedule and uses a small profile set. Set `RUN_OPTIONAL_DEMOS = True` to run it.


In [ ]:
forgiveness_results = run_demo("forgiveness")
